# Redes neurais, passo a passo

Este notebook tem duas partes.

1. **O perceptron.** Um neurônio só, doze pontos e uma reta. Cada célula faz **uma** atualização dos pesos e desenha como a reta ficou.
2. **A camada oculta.** Um conjunto de pontos em que a classe 1 ocupa duas áreas separadas e de contorno curvo. Sem camada oculta, a rede não separa as áreas; com mais neurônios na camada oculta, a separação melhora.

Execute as células na ordem, uma por vez (`Shift+Enter`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

AZUL, CEU, NAVY, LARANJA, CINZA = "#2264AF", "#8FCEF1", "#27316E", "#D9480F", "#8A99A5"

plt.rcParams.update({
    "figure.figsize": (6, 6), "figure.dpi": 100, "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": "#DCE5EE", "grid.linewidth": 0.8,
})
np.set_printoptions(precision=2, suppress=True)
pd.set_option("display.precision", 2)

## Parte 1: o perceptron

### Os dados

Doze pontos com duas entradas, $x_1$ e $x_2$, e uma classe $y$, que vale 0 ou 1. Os pontos saem de um sorteio com semente fixa, para que os números sejam os mesmos em toda execução.

In [ ]:
rng = np.random.default_rng(7)

pontos_classe_0 = rng.normal(loc=[1.5, 3.5], scale=0.6, size=(6, 2))
pontos_classe_1 = rng.normal(loc=[3.5, 1.5], scale=0.6, size=(6, 2))

X = np.vstack([pontos_classe_0, pontos_classe_1]).round(1)
y = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1])

dados = pd.DataFrame({"x1": X[:, 0], "x2": X[:, 1], "y": y})
dados.index.name = "ponto"
dados

### O neurônio

O perceptron faz duas contas com cada ponto:

$$z = w_1 x_1 + w_2 x_2 + b \qquad\qquad \hat{y} = \begin{cases} 1 & \text{se } z \ge 0 \\ 0 & \text{se } z < 0 \end{cases}$$

Os pontos em que $z = 0$ formam uma **reta**. De um lado dela o neurônio responde 1; do outro, 0. Aprender é mover essa reta, e quem a move são os pesos $w_1$, $w_2$ e o viés $b$.

In [ ]:
def degrau(z):
    return (z >= 0).astype(int)


def desenhar(X, y, w, b, titulo, w_antes=None, b_antes=None, destaque=None):
    # Região em que o neurônio responde 1 (azul claro), a reta z = 0 (laranja)
    # e, se pedido, a reta de antes da atualização (cinza tracejada).
    eixo = np.linspace(0, 5, 300)
    G1, G2 = np.meshgrid(eixo, eixo)

    fig, ax = plt.subplots()
    Z = w[0] * G1 + w[1] * G2 + b
    ax.contourf(G1, G2, Z, levels=[0, np.inf], colors=[CEU], alpha=0.45)
    if w_antes is not None:
        Z_antes = w_antes[0] * G1 + w_antes[1] * G2 + b_antes
        ax.contour(G1, G2, Z_antes, levels=[0], colors=[CINZA], linewidths=2, linestyles="--")
    ax.contour(G1, G2, Z, levels=[0], colors=[LARANJA], linewidths=3)

    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=150, facecolor="white", edgecolor=NAVY,
               linewidth=2.5, zorder=3, label="classe 0")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=150, color=NAVY, zorder=3, label="classe 1")

    previsoes = degrau(X @ w + b)
    errados = previsoes != y
    ax.scatter(X[errados, 0], X[errados, 1], marker="x", s=90, color=LARANJA,
               linewidth=2.5, zorder=4, label="ponto errado")
    if destaque is not None:
        ax.scatter(X[destaque, 0], X[destaque, 1], s=520, facecolor="none", edgecolor=LARANJA,
                   linewidth=3, zorder=5, label="ponto usado na atualização")
    for i in range(len(X)):
        ax.annotate(str(i), X[i], xytext=(9, 9), textcoords="offset points", color=NAVY, fontsize=10)

    ax.set(xlim=(0, 5), ylim=(0, 5), xlabel="x1", ylabel="x2", title=titulo, aspect="equal")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
    plt.show()

### O ponto de partida

Os pesos começam com valores quaisquer. No gráfico, a área azul é onde o neurônio responde 1, a linha laranja é a reta $z = 0$, e um **×** marca cada ponto classificado errado.

In [ ]:
w = np.array([-1.0, 1.0])
b = 0.0
taxa = 0.2

historico = []   # uma linha por atualização, para a tabela do fim

desenhar(X, y, w, b, "Antes de qualquer atualização")

A tabela abaixo faz a conta do neurônio para cada ponto, com os pesos iniciais.

In [ ]:
z = X @ w + b
previsao = degrau(z)

tabela = dados.copy()
tabela["z"] = z
tabela["previsão"] = previsao
tabela["resultado"] = np.where(previsao == y, "certo", "errado")
print("pontos errados:", int((previsao != y).sum()), "de", len(y))
tabela

### A regra de atualização

Pega-se **um ponto errado** e calcula-se o erro dele, $y - \hat{y}$, que vale $+1$ ou $-1$. Os pesos andam nessa direção:

$$w \leftarrow w + \eta \,(y - \hat{y})\, x \qquad\qquad b \leftarrow b + \eta \,(y - \hat{y})$$

O número $\eta$ é a taxa de aprendizado (aqui, `taxa = 0.2`). Se o ponto é da classe 1 e o neurônio disse 0, o erro é $+1$ e os pesos somam um pedaço de $x$, o que aumenta o $z$ daquele ponto. Se é da classe 0 e o neurônio disse 1, o erro é $-1$ e o $z$ diminui.

### Atualização 1

A primeira vai por extenso, uma conta por linha. A reta cinza tracejada é a de antes; a laranja é a nova; o círculo laranja marca o ponto usado.

In [ ]:
# 1. Quais pontos estão errados com os pesos de agora?
previsoes = degrau(X @ w + b)
errados = np.flatnonzero(previsoes != y)
print("pontos errados:", errados)

# 2. Pega o primeiro deles.
i = errados[0]
x_i = X[i]
print("ponto usado:", i, "| x =", x_i, "| y =", y[i])

# 3. A conta do neurônio para esse ponto.
z_i = w @ x_i + b
previsao_i = degrau(z_i)
erro_i = y[i] - previsao_i
print("z =", round(z_i, 2), "| previsão =", previsao_i, "| erro (y - previsão) =", erro_i)

# 4. A atualização.
w_novo = w + taxa * erro_i * x_i
b_novo = b + taxa * erro_i
print("w antes:", w, "| w depois:", w_novo)
print("b antes:", round(b, 2), "| b depois:", round(b_novo, 2))

# 5. Quantos pontos a reta nova erra?
erros_depois = int((degrau(X @ w_novo + b_novo) != y).sum())
print("pontos errados depois:", erros_depois)

desenhar(X, y, w_novo, b_novo, "Atualização 1", w_antes=w, b_antes=b, destaque=i)

historico.append({"atualização": 1, "ponto usado": i, "erro": erro_i, "w1": w_novo[0],
                  "w2": w_novo[1], "b": b_novo, "pontos errados depois": erros_depois})
w, b = w_novo, b_novo

### Atualização 2

As mesmas cinco contas, agora partindo dos pesos que a atualização 1 deixou.

In [ ]:
# 1. Quais pontos estão errados com os pesos de agora?
previsoes = degrau(X @ w + b)
errados = np.flatnonzero(previsoes != y)
print("pontos errados:", errados)

# 2. Pega o primeiro deles.
i = errados[0]
x_i = X[i]
print("ponto usado:", i, "| x =", x_i, "| y =", y[i])

# 3. A conta do neurônio para esse ponto.
z_i = w @ x_i + b
previsao_i = degrau(z_i)
erro_i = y[i] - previsao_i
print("z =", round(z_i, 2), "| previsão =", previsao_i, "| erro (y - previsão) =", erro_i)

# 4. A atualização.
w_novo = w + taxa * erro_i * x_i
b_novo = b + taxa * erro_i
print("w antes:", w, "| w depois:", w_novo)
print("b antes:", round(b, 2), "| b depois:", round(b_novo, 2))

# 5. Quantos pontos a reta nova erra?
erros_depois = int((degrau(X @ w_novo + b_novo) != y).sum())
print("pontos errados depois:", erros_depois)

desenhar(X, y, w_novo, b_novo, "Atualização 2", w_antes=w, b_antes=b, destaque=i)

historico.append({"atualização": 2, "ponto usado": i, "erro": erro_i, "w1": w_novo[0],
                  "w2": w_novo[1], "b": b_novo, "pontos errados depois": erros_depois})
w, b = w_novo, b_novo

### As próximas atualizações

A conta não muda mais. A função abaixo guarda as mesmas cinco etapas, e cada célula seguinte a chama **uma vez**: uma célula, uma atualização, um gráfico.

Em algumas atualizações a reta laranja não aparece: ela foi parar fora do quadro, e o neurônio responde a mesma classe para os doze pontos. O gráfico fica todo azul (tudo 1) ou todo branco (tudo 0).

In [ ]:
def atualizar(X, y, w, b, taxa, historico):
    numero = len(historico) + 1

    previsoes = degrau(X @ w + b)
    errados = np.flatnonzero(previsoes != y)
    if len(errados) == 0:
        print("Nenhum ponto errado: não há o que atualizar. O perceptron parou.")
        desenhar(X, y, w, b, "A reta final, depois de " + str(len(historico)) + " atualizações")
        return w, b
    print("pontos errados:", errados)

    i = errados[0]
    x_i = X[i]
    z_i = w @ x_i + b
    previsao_i = degrau(z_i)
    erro_i = y[i] - previsao_i
    print("ponto usado:", i, "| x =", x_i, "| y =", y[i])
    print("z =", round(z_i, 2), "| previsão =", previsao_i, "| erro (y - previsão) =", erro_i)

    w_novo = w + taxa * erro_i * x_i
    b_novo = b + taxa * erro_i
    print("w antes:", w, "| w depois:", w_novo)
    print("b antes:", round(b, 2), "| b depois:", round(b_novo, 2))

    erros_depois = int((degrau(X @ w_novo + b_novo) != y).sum())
    print("pontos errados depois:", erros_depois)

    desenhar(X, y, w_novo, b_novo, "Atualização " + str(numero), w_antes=w, b_antes=b, destaque=i)

    historico.append({"atualização": numero, "ponto usado": i, "erro": erro_i, "w1": w_novo[0],
                      "w2": w_novo[1], "b": b_novo, "pontos errados depois": erros_depois})
    return w_novo, b_novo

### Atualização 3

In [ ]:
w, b = atualizar(X, y, w, b, taxa, historico)

### Atualização 4

In [ ]:
w, b = atualizar(X, y, w, b, taxa, historico)

### Atualização 5

In [ ]:
w, b = atualizar(X, y, w, b, taxa, historico)

### Atualização 6

In [ ]:
w, b = atualizar(X, y, w, b, taxa, historico)

### Atualização 7

In [ ]:
w, b = atualizar(X, y, w, b, taxa, historico)

### Atualização 8

In [ ]:
w, b = atualizar(X, y, w, b, taxa, historico)

### Mais uma chamada

Antes de executar a próxima célula, conte os **×** no último gráfico.

In [ ]:
w, b = atualizar(X, y, w, b, taxa, historico)

### O caminho inteiro

Cada linha da tabela é uma das atualizações acima: o ponto usado, o sinal do erro, os pesos que ficaram e quantos pontos a reta ainda errava.

In [ ]:
caminho = pd.DataFrame(historico).set_index("atualização")
caminho

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(caminho.index, caminho["pontos errados depois"], marker="o", color=LARANJA, linewidth=2)
ax.set(xlabel="atualização", ylabel="pontos errados", title="Pontos errados depois de cada atualização",
       xticks=caminho.index, ylim=(-0.5, len(y) + 0.5))
plt.show()

Uma atualização conserta o ponto usado ou o aproxima do lado certo, mas pode estragar outros: o número de pontos errados não precisa cair a cada passo. Quando existe uma reta que separa as duas classes, como aqui, o perceptron sempre chega a uma delas depois de um número finito de atualizações.

**Para experimentar:** volte à célula do ponto de partida, troque `taxa` ou os pesos iniciais e execute tudo de novo. A reta final é a mesma? O número de atualizações é o mesmo?

## Parte 2: quando uma reta não basta

### Os dados

Agora são 700 pontos espalhados num quadrado. A classe 1 ocupa **duas áreas que não se tocam**: um disco, em cima à esquerda, e uma faixa curva, embaixo à direita. Todo o resto é classe 0.

In [ ]:
rng = np.random.default_rng(7)
X2 = rng.uniform(low=-3, high=3, size=(700, 2))
x1 = X2[:, 0]
x2 = X2[:, 1]

# Área 1: um disco de raio 1 com centro em (-1,5; 1,5).
no_disco = (x1 + 1.5) ** 2 + (x2 - 1.5) ** 2 < 1.0

# Área 2: uma faixa em volta da parábola x2 = 0,5 (x1 - 0,8)² - 2.
parabola = 0.5 * (x1 - 0.8) ** 2 - 2.0
na_faixa = (np.abs(x2 - parabola) < 0.5) & (np.abs(x1 - 0.8) < 1.8)

y2 = (no_disco | na_faixa).astype(int)

contagem = pd.DataFrame({"pontos": [int((y2 == 0).sum()), int((y2 == 1).sum())]}, index=["classe 0", "classe 1"])
contagem["proporção"] = contagem["pontos"] / len(y2)
contagem

In [ ]:
def desenhar_pontos(ax, X, y):
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=22, facecolor="white", edgecolor=NAVY, linewidth=1, label="classe 0")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=22, color=NAVY, label="classe 1")
    ax.set(xlim=(-3, 3), ylim=(-3, 3), xlabel="x1", ylabel="x2", aspect="equal")
    ax.grid(False)


fig, ax = plt.subplots()
desenhar_pontos(ax, X2, y2)
ax.set_title("A classe 1 ocupa duas áreas separadas")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=2, frameon=False)
plt.show()

Nenhuma reta deixa o disco e a faixa de um lado e todo o resto do outro.

### Sem camada oculta

A rede mais simples liga as duas entradas direto à saída: é um neurônio só, como o perceptron, com a sigmoide no lugar do degrau. No `scikit-learn`, isso é um `MLPClassifier` com `hidden_layer_sizes=()`, a lista vazia de camadas ocultas.

Primeiro monta-se a rede, depois ela é ajustada aos pontos, depois prevê a classe de cada ponto, e por fim conta-se quantos acertou.

In [ ]:
from sklearn.neural_network import MLPClassifier

# 1. Montar a rede: nenhuma camada oculta.
rede_sem_oculta = MLPClassifier(hidden_layer_sizes=(), solver="lbfgs", max_iter=20000, random_state=0)

# 2. Ajustar aos pontos.
rede_sem_oculta.fit(X2, y2)

# 3. Prever a classe de cada ponto.
previsao_sem_oculta = rede_sem_oculta.predict(X2)

# 4. Contar os acertos.
acertos = int((previsao_sem_oculta == y2).sum())
print("pontos acertados:", acertos, "de", len(y2))
print("proporção de acertos:", round(acertos / len(y2), 3))
print("pontos em que a rede respondeu 1:", int(previsao_sem_oculta.sum()))

Compare a proporção de acertos com a proporção da classe 0 na tabela de contagem, e veja em quantos pontos a rede respondeu 1. O gráfico mostra a resposta da rede em cada lugar do quadrado: azul onde ela responde 1, branco onde responde 0.

In [ ]:
def desenhar_regiao(ax, rede, X, y, titulo):
    # Pinta de azul a região em que a rede responde 1 e traça a fronteira em laranja.
    eixo = np.linspace(-3, 3, 300)
    G1, G2 = np.meshgrid(eixo, eixo)
    grade = np.column_stack([G1.ravel(), G2.ravel()])
    P = rede.predict_proba(grade)[:, 1].reshape(G1.shape)
    ax.contourf(G1, G2, P, levels=[0.5, 1.01], colors=[CEU], alpha=0.6)
    if P.min() < 0.5 < P.max():
        ax.contour(G1, G2, P, levels=[0.5], colors=[LARANJA], linewidths=2.5)
    desenhar_pontos(ax, X, y)
    ax.set_title(titulo)


fig, ax = plt.subplots()
desenhar_regiao(ax, rede_sem_oculta, X2, y2, "Sem camada oculta")
plt.show()

Sem camada oculta, a fronteira só pode ser uma reta, e nenhuma reta serve aqui. O melhor que o ajuste encontra é responder a classe mais comum.

### Uma camada oculta com 3 neurônios

Cada neurônio da camada oculta calcula o seu próprio $z$, com a sua própria reta, e a saída combina as respostas deles. As etapas são as mesmas; só muda `hidden_layer_sizes`.

In [ ]:
# 1. Montar a rede: uma camada oculta com 3 neurônios.
rede_3 = MLPClassifier(hidden_layer_sizes=(3,), activation="tanh", solver="lbfgs", max_iter=20000, random_state=0)

# 2. Ajustar aos pontos.
rede_3.fit(X2, y2)

# 3. Prever a classe de cada ponto.
previsao_3 = rede_3.predict(X2)

# 4. Contar os acertos.
acertos_3 = int((previsao_3 == y2).sum())
print("pontos acertados:", acertos_3, "de", len(y2))
print("proporção de acertos:", round(acertos_3 / len(y2), 3))
print("pontos em que a rede respondeu 1:", int(previsao_3.sum()))

fig, ax = plt.subplots()
desenhar_regiao(ax, rede_3, X2, y2, "Camada oculta com 3 neurônios")
plt.show()

### O que cada neurônio oculto faz

Os pesos da primeira camada ficam em `rede_3.coefs_[0]`, uma coluna por neurônio oculto, e os vieses em `rede_3.intercepts_[0]`. Cada neurônio oculto tem, portanto, a sua reta $z = 0$, como o perceptron da parte 1.

In [ ]:
pesos_ocultos = rede_3.coefs_[0]        # 2 linhas (x1, x2) e 3 colunas (um neurônio por coluna)
vieses_ocultos = rede_3.intercepts_[0]

tabela_ocultos = pd.DataFrame({
    "w1": pesos_ocultos[0],
    "w2": pesos_ocultos[1],
    "b": vieses_ocultos,
}, index=["neurônio 1", "neurônio 2", "neurônio 3"])
tabela_ocultos

In [ ]:
eixo = np.linspace(-3, 3, 300)
G1, G2 = np.meshgrid(eixo, eixo)
cores = [LARANJA, "#2B8A3E", "#7048E8"]

fig, ax = plt.subplots()
desenhar_regiao(ax, rede_3, X2, y2, "As três retas da camada oculta")
for j in range(3):
    Z = pesos_ocultos[0, j] * G1 + pesos_ocultos[1, j] * G2 + vieses_ocultos[j]
    ax.contour(G1, G2, Z, levels=[0], colors=[cores[j]], linewidths=2, linestyles="--")
    ax.plot([], [], color=cores[j], linestyle="--", linewidth=2, label="neurônio " + str(j + 1))
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, frameon=False)
plt.show()

A área azul é montada com as retas tracejadas: a saída da rede soma as respostas dos três neurônios, cada uma mudando de sinal ao cruzar a sua reta. Com três retas, o desenho possível ainda é pobre para um disco e uma faixa curva.

### Mais neurônios na camada oculta

A mesma conta, para cada tamanho de camada oculta: montar, ajustar, prever, contar os acertos.

In [ ]:
tamanhos = [0, 2, 3, 5, 10, 20]
redes = {}
linhas = []

for n_ocultos in tamanhos:
    if n_ocultos == 0:
        camadas = ()
    else:
        camadas = (n_ocultos,)

    rede = MLPClassifier(hidden_layer_sizes=camadas, activation="tanh", solver="lbfgs", max_iter=20000, random_state=0)
    rede.fit(X2, y2)
    previsao = rede.predict(X2)

    acertos = int((previsao == y2).sum())
    redes[n_ocultos] = rede
    linhas.append({
        "neurônios ocultos": n_ocultos,
        "pontos acertados": acertos,
        "pontos errados": len(y2) - acertos,
        "proporção de acertos": acertos / len(y2),
    })

resultado = pd.DataFrame(linhas).set_index("neurônios ocultos")
resultado

In [ ]:
fig, eixos = plt.subplots(2, 3, figsize=(15, 10.5))

for ax, n_ocultos in zip(eixos.ravel(), tamanhos):
    erros = resultado.loc[n_ocultos, "pontos errados"]
    titulo = str(n_ocultos) + " neurônios ocultos: " + str(erros) + " pontos errados"
    desenhar_regiao(ax, redes[n_ocultos], X2, y2, titulo)

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.8))
posicoes = range(len(tamanhos))
ax.bar(posicoes, resultado["pontos errados"], color=AZUL, width=0.6)
for posicao, erros in zip(posicoes, resultado["pontos errados"]):
    ax.annotate(str(erros), (posicao, erros), xytext=(0, 4), textcoords="offset points", ha="center")
ax.set(xticks=list(posicoes), xticklabels=tamanhos, xlabel="neurônios na camada oculta",
       ylabel="pontos errados", title="Pontos errados, por tamanho da camada oculta")
ax.grid(axis="x", visible=False)
plt.show()

Leia os painéis na ordem. Sem camada oculta, não há área azul. Com poucos neurônios, a rede desenha uma área só, de contorno grosseiro. Com mais neurônios, há retas suficientes para contornar o disco e a faixa, e as duas áreas aparecem separadas.

Uma ressalva sobre o que foi medido: os acertos acima são contados nos **mesmos pontos** usados no ajuste. Isso mostra o que a rede é **capaz de desenhar**, e não como ela se sairia em pontos novos. Uma rede grande o bastante também contorna ruído; para escolher o tamanho da camada, o erro tem de ser medido em pontos que ficaram fora do ajuste.

### Para experimentar

1. Troque `random_state=0` por outro número no laço. Os pesos iniciais mudam; a rede de 5 neurônios chega sempre ao mesmo desenho?
2. Troque `activation="tanh"` por `"relu"`. Como fica o contorno das áreas?
3. Use duas camadas ocultas pequenas, `hidden_layer_sizes=(5, 5)`, e compare com uma camada de 10.
4. Mude os dados: aumente o raio do disco, ou acrescente uma terceira área à classe 1. Quantos neurônios passam a ser necessários?